# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata object directly, no subscripting
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"Published on: {dataset.metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# Get record sets from dataset
record_sets = dataset.metadata.recordSets

print("Available record sets:")
for rs in record_sets:
    print(f"- name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print(f"  Description: {getattr(rs, 'description', 'No description')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field['@id']}) [type: {field.dataType}]")
    print()

## 3. Data Extraction

Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the above overview.

**Note:** Replace `<record_set_id>` and `<field_id>` below with actual `@id`s as identified.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

# We'll create a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record set @ids:", record_set_ids)

for record_set_id in record_set_ids:
    # Load the records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from {record_set_id}, columns: {df.columns.tolist()}")

# Choose a record set to display (here, take the first one as example)
main_record_set_id = record_set_ids[0]
print(f"Sample records from {main_record_set_id}:")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** For demonstration, identify a numeric field and a grouping field using their `@id` (see above overview).

In [ ]:
# Example: Use 'gad7_score' and 'village' fields by their @id.
# Update these IDs based on actual @ids from overview.

# Find relevant field @ids
rs_fields = record_sets[0].fields
numeric_fields = [field for field in rs_fields if field.dataType in ['Float', 'Integer', 'Number']]
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]['@id']
else:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes(include=['number']).columns[0]

group_field = None
for field in rs_fields:
    if field.dataType == 'Text' and 'village' in field.name.lower():
        group_field = field['@id']
if not group_field:
    # Try 'Village' or choose first text field
    group_field = next((field['@id'] for field in rs_fields if field.dataType == 'Text'), None)

print(f"Numeric field chosen (@id): {numeric_field_id}")
print(f"Group field chosen (@id): {group_field}")

# Check these fields are in the dataframe
df = dataframes[main_record_set_id]
if numeric_field_id not in df.columns or group_field not in df.columns:
    print("Unable to find fields for EDA. Explore columns:")
    print(df.head())
else:
    threshold = df[numeric_field_id].quantile(0.75) # Use upper quartile as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by grouping field (e.g., village)
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped means by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field and group means per village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the filtered distribution
if numeric_field_id in df.columns and group_field in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(filtered_df[numeric_field_id], bins=20, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id} (> {threshold:.2f})")
    axes[0].set_xlabel(numeric_field_id)

    # Barplot of mean by group
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df, ax=axes[1])
    axes[1].set_title(f"Mean {numeric_field_id} by {group_field}")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped: fields not found.")

## 6. Conclusion

- We loaded the FAIR² Mental Health Survey Dataset using `mlcroissant`, referencing all entities by their `@id`.
- Reviewed available record sets and fields, extracted data, and performed basic exploratory analysis of numeric indicators (e.g., GAD-7 scores).
- Performed grouping by demographic attributes such as village and visualized data distributions.
- The notebook provides a reproducible template for Croissant datasets, facilitating further analysis for mental health and demographic factors.

If you wish to extend this notebook, refer to further documentation of the [mlcroissant API](https://mlcroissant.readthedocs.io/) or the [Croissant schema](https://github.com/mlcommons/croissant).